# 04 – Comparing the groups

**Input:** `results/peptide_features.csv` (from notebook 03)  
**Outputs:** figures in `figures/` and tables `results/comparison_*.csv`

All comparisons use the **non-redundant set**: only the CD-HIT representatives (90% identity, notebook 02). Section 1c shows how much this changes the answer.

We answer three questions:

* **Q1** – How do *Natural* and *Synthetic* peptides differ in length, charge and hydropathy?
* **Q2** – How do peptides listed with different target groups (Gram+, Gram-, fungi, virus, cancer) differ?
* **Q3** – Do peptides listed with *mammalian cells* as a target look different from the others?

(The amino-acid composition comparison is in notebook 05.)

## Methods and where they come from
| Step | Method | Source |
|---|---|---|
| two groups | **Mann-Whitney U test** (rank-based, for two *independent* groups) | Mann & Whitney (1947); description of the test in Fiel Peres (2025) |
| more than two groups | **Kruskal-Wallis test** (extension of Mann-Whitney to more than two groups) | Kruskal & Wallis (1952); Fiel Peres (2025) |
| effect size, two groups | **Cliff's delta (δ)** = rank-biserial correlation = 2 × (Vargha-Delaney A) − 1, where A = U / (n₁·n₂). 0 = no difference, ±1 = the groups do not overlap at all | Fiel Peres (2025), eqs. 3-5 |
| effect-size labels | \|δ\| ≥ 0.11 small, ≥ 0.28 medium, ≥ 0.43 large (below 0.11 we write "negligible") | thresholds of Vargha & Delaney as reported by Fiel Peres (2025). The review stresses they are only heuristics, and that other authors use 0.1 / 0.3 / 0.5 |
| effect size, Kruskal-Wallis | **ordinal eta-squared** η²ₕ = (H − k + 1) / (N − k) (H = test statistic, k = groups, N = peptides); small ≥ 0.01, medium ≥ 0.06, large ≥ 0.14 | Fiel Peres (2025) |
| after Kruskal-Wallis | pairwise Cliff's delta for every pair of groups | Fiel Peres (2025): "calculate the effect size for all possible pairwise comparisons" |

**Why effect sizes and not only p-values?** With thousands of peptides almost every difference is "significant"; p-values depend strongly on sample size, so effect sizes are reported alongside them (Fiel Peres 2025).

**Features** are defined in notebook 03 (Biopython net charge at pH 7, charge per residue, mean Kyte-Doolittle hydropathy "GRAVY").

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.stats import mannwhitneyu, kruskal

allp = pd.read_csv("../results/peptide_features.csv")     # every cleaned peptide (before CD-HIT)
df = allp[allp["is_representative"]].copy()              # non-redundant set used for all analyses
FIG, RES = "../figures", "../results"

FEATURES = ["length", "net_charge", "charge_per_residue", "gravy"]
TITLES = {"length": "Length (residues)", "net_charge": "Net charge (pH 7)",
          "charge_per_residue": "Charge per residue", "gravy": "Mean hydropathy (GRAVY)"}
COLORS = {"Natural": "#2a9d8f", "Synthetic": "#e76f51"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"{len(allp)} cleaned peptides -> {len(df)} non-redundant representatives")

18275 cleaned peptides -> 9951 non-redundant representatives


## Helper functions
Small and reusable: one effect-size function, one label function, one comparison function.

In [2]:
def cliffs_delta(a, b):
    """Cliff's delta = 2*U/(n1*n2) - 1 (= 2*A - 1, A = Vargha-Delaney A); see Fiel Peres (2025).
    U is the Mann-Whitney U statistic of the first sample (Mann & Whitney 1947)."""
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    return 2 * u / (len(a) * len(b)) - 1, p

def size_label(d):
    """Vargha-Delaney thresholds for |delta| as reported by Fiel Peres (2025)."""
    d = abs(d)
    return "negligible" if d < 0.11 else "small" if d < 0.28 else "medium" if d < 0.43 else "large"

def eta_squared_H(H, k, N):
    """Ordinal eta-squared for the Kruskal-Wallis test: (H - k + 1) / (N - k); see Fiel Peres (2025)."""
    return (H - k + 1) / (N - k)

def eta_label(e):
    return "negligible" if e < 0.01 else "small" if e < 0.06 else "medium" if e < 0.14 else "large"

def compare_two(data, group_col, g1, g2, features):
    """Mann-Whitney + Cliff's delta for each feature, g1 vs g2."""
    a, b = data[data[group_col] == g1], data[data[group_col] == g2]
    rows = []
    for f in features:
        d, p = cliffs_delta(a[f], b[f])
        rows.append({"feature": f, f"median_{g1}": a[f].median(), f"median_{g2}": b[f].median(),
                     "cliffs_delta": round(d, 3), "effect": size_label(d), "p_value": p})
    return pd.DataFrame(rows)

# tiny self-tests: fully separated groups -> delta = +1 / -1, identical groups -> 0
assert cliffs_delta(pd.Series([5, 6, 7]), pd.Series([1, 2, 3]))[0] == 1
assert cliffs_delta(pd.Series([1, 2, 3]), pd.Series([5, 6, 7]))[0] == -1
assert abs(cliffs_delta(pd.Series([1, 2, 3]), pd.Series([1, 2, 3]))[0]) < 1e-9
# worked example of Fiel Peres (2025): H = 8.4, k = 3 groups, N = 30 -> eta-squared(H) = 0.237
assert abs(eta_squared_H(8.4, 3, 30) - 0.237) < 0.001
print("helpers OK")

helpers OK


# Q1 – Natural vs Synthetic
## 1a. Sample sizes (non-redundant set)

In [3]:
df["origin"].value_counts()

origin
Synthetic    7650
Natural      2301
Name: count, dtype: int64

## 1b. Length, charge and hydropathy

In [4]:
q1 = compare_two(df, "origin", "Natural", "Synthetic", FEATURES)
q1.to_csv(f"{RES}/comparison_q1_origin.csv", index=False)
q1.round({"median_Natural": 2, "median_Synthetic": 2}).style.format({"p_value": "{:.1e}"})

,feature,median_Natural,median_Synthetic,cliffs_delta,effect,p_value
0,length,25.000000,14.000000,0.487000,large,2.4e-276
1,net_charge,2.680000,3.760000,-0.257000,small,5.2e-78
2,charge_per_residue,0.090000,0.270000,-0.588000,large,0.0e+00
3,gravy,0.170000,-0.350000,0.318000,medium,2.2e-118


In [5]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, f in zip(axes, FEATURES):
    data = [df.loc[df.origin == o, f] for o in ["Natural", "Synthetic"]]
    bp = ax.boxplot(data, tick_labels=["Natural", "Synthetic"], showfliers=False, patch_artist=True,
                    medianprops={"color": "black"})
    for patch, o in zip(bp["boxes"], ["Natural", "Synthetic"]):
        patch.set_facecolor(COLORS[o]); patch.set_alpha(0.8)
    d = q1.set_index("feature").loc[f]
    ax.set_title(f"{TITLES[f]}\nδ = {d.cliffs_delta:+.2f} ({d.effect})", fontsize=10)
fig.suptitle("Natural vs Synthetic peptides (outliers hidden)", y=1.02)
plt.tight_layout()
plt.savefig(f"{FIG}/q1_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 1c. Does removing redundancy change the answer?
The same comparison on all cleaned peptides (before CD-HIT) and on the non-redundant representatives.

In [6]:
rows = []
for label, data in [("before CD-HIT", allp), ("after CD-HIT", df)]:
    r = compare_two(data, "origin", "Natural", "Synthetic", FEATURES)
    r.insert(0, "dataset", label)
    r.insert(1, "n_nat/n_syn", f"{(data.origin=='Natural').sum()}/{(data.origin=='Synthetic').sum()}")
    rows.append(r)
red = pd.concat(rows, ignore_index=True)
red.to_csv(f"{RES}/comparison_q1_redundancy.csv", index=False)
red.pivot(index="feature", columns="dataset", values="cliffs_delta")[["before CD-HIT", "after CD-HIT"]].loc[FEATURES]

dataset,before CD-HIT,after CD-HIT
feature,,
length,0.444,0.487
net_charge,-0.282,-0.257
charge_per_residue,-0.587,-0.588
gravy,0.340,0.318


## 1d. Robustness: is it just because natural peptides are longer?
Natural peptides are longer than synthetic ones, and length can influence the other features. As a sensitivity check we repeat the comparison (i) using only peptides of the length class *Short (10–24 aa)* defined in this repository's `by_length_class` files, so both groups have similar lengths, and (ii) using only peptides without D-amino acids, which are more common among synthetic peptides.

In [7]:
subsets = {
    "all peptides": df,
    "Short (10-24 aa) only": df[df.length_class == "Short (10–24 aa)"],
    "no D-amino acids": df[~df.has_D_aa],
}
rows = []
for name, sub in subsets.items():
    res = compare_two(sub, "origin", "Natural", "Synthetic", ["length", "charge_per_residue", "gravy"])
    res.insert(0, "subset", name)
    res.insert(1, "n_nat/n_syn", f"{(sub.origin=='Natural').sum()}/{(sub.origin=='Synthetic').sum()}")
    rows.append(res)
robust = pd.concat(rows, ignore_index=True)
robust.to_csv(f"{RES}/comparison_q1_robustness.csv", index=False)
robust.pivot(index="feature", columns="subset", values="cliffs_delta")[list(subsets)]

subset,all peptides,Short (10-24 aa) only,no D-amino acids
feature,,,
charge_per_residue,-0.588,-0.661,-0.577
gravy,0.318,0.511,0.307
length,0.487,0.279,0.510


## Amino-acid composition
The amino-acid comparison (against a background population, with enrichment statistics) is in notebook **05_enrichment**.

## 1e. Which target groups are the peptides listed with?
Percentage of peptides in each origin that are *listed with* each target group (groups overlap, so columns do not add to 100%). DBAASP includes peptides that were tested regardless of outcome (Pirtskhalava et al. 2021), so this is not necessarily "active against". Only peptides with target information are used.

In [8]:
target_cols = ["target_gram_pos", "target_gram_neg", "target_fungus", "target_cancer", "target_virus",
               "target_biofilm", "target_parasite", "target_mammalian_cell"]
t = df[df.has_target_info]
share = (t.groupby("origin")[target_cols].mean() * 100).T
share.index = [i.replace("target_", "") for i in share.index]
counts_by_origin = t.groupby("origin")[target_cols].sum().T
share["n_natural"] = counts_by_origin["Natural"].values
share["n_synthetic"] = counts_by_origin["Synthetic"].values
share.index.name = "target_group"
share.to_csv(f"{RES}/comparison_q1_targets.csv")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
im = ax.imshow(share[["Natural", "Synthetic"]].values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Natural", "Synthetic"])
ax.set_yticks(range(len(share))); ax.set_yticklabels(share.index)
for i in range(len(share)):
    for j, o in enumerate(["Natural", "Synthetic"]):
        v = share.loc[share.index[i], o]
        ax.text(j, i, f"{v:.0f}%", ha="center", va="center", color="white" if v > 55 else "black")
ax.set_title("% of peptides listed with each target group", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG}/q1_target_heatmap.png", dpi=150)
plt.show()
share.round(1)

origin,Natural,Synthetic,n_natural,n_synthetic
target_group,,,,
gram_pos,86.7,78.0,1982,5877
gram_neg,84.6,80.8,1934,6094
fungus,54.3,22.3,1241,1683
cancer,20.2,18.0,461,1358
virus,4.4,7.7,101,581
biofilm,2.2,3.9,51,294
parasite,5.0,0.9,115,71
mammalian_cell,52.4,56.7,1198,4271


# Q2 – Peptides listed with different target groups
Target groups overlap (a peptide can be listed with Gram+, Gram- and fungi at once). The Mann-Whitney and Kruskal-Wallis tests compare *independent* groups (Fiel Peres 2025), so overlapping groups would break their assumption. To keep the groups **independent**, we only use peptides whose target list is *exactly* one of these sets:

| Group | Meaning |
|---|---|
| Gram+ only | listed only with Gram-positive bacteria |
| Gram- only | listed only with Gram-negative bacteria |
| Gram+ & Gram- | listed with both bacterial groups, nothing else |
| Fungus only | listed only with fungi |
| Virus only | listed only with viruses |
| Cancer only | listed only with cancer cells |

In [9]:
TARGETS = [c for c in df.columns if c.startswith("target_")]      # the 13 yes/no target columns from notebook 01
t = df[df.has_target_info].copy()

EXCLUSIVE = {frozenset(["target_gram_pos"]): "Gram+ only", frozenset(["target_gram_neg"]): "Gram- only",
             frozenset(["target_gram_pos", "target_gram_neg"]): "Gram+ & Gram-",
             frozenset(["target_fungus"]): "Fungus only", frozenset(["target_virus"]): "Virus only",
             frozenset(["target_cancer"]): "Cancer only"}
t["target_group"] = t.apply(lambda r: EXCLUSIVE.get(frozenset(c for c in TARGETS if r[c] == 1)), axis=1)
t = t.dropna(subset=["target_group"])
GROUPS = ["Gram+ only", "Gram- only", "Gram+ & Gram-", "Fungus only", "Virus only", "Cancer only"]
print(t["target_group"].value_counts().reindex(GROUPS))
print("\nOrigin mix inside each group (% Natural):")
print((t.groupby("target_group")["origin"].apply(lambda s: (s == "Natural").mean() * 100)).reindex(GROUPS).round(1))

target_group
Gram+ only        435
Gram- only        308
Gram+ & Gram-    1673
Fungus only       192
Virus only        385
Cancer only       122
Name: count, dtype: int64

Origin mix inside each group (% Natural):
target_group
Gram+ only       16.6
Gram- only       13.0
Gram+ & Gram-    21.2
Fungus only      31.2
Virus only        1.3
Cancer only      52.5
Name: origin, dtype: float64


## 2a. Overall difference between the groups (Kruskal-Wallis)

In [10]:
rows = []
N, k = len(t), len(GROUPS)
for f in FEATURES:
    H, p = kruskal(*[t.loc[t.target_group == g, f] for g in GROUPS])
    eta = eta_squared_H(H, k, N)
    rows.append({"feature": f, "H": round(H, 1), "eta_sq_H": round(eta, 3), "effect": eta_label(eta), "p_value": p,
                 **{f"median_{g}": round(t.loc[t.target_group == g, f].median(), 2) for g in GROUPS}})
q2 = pd.DataFrame(rows)
q2.to_csv(f"{RES}/comparison_q2_targets.csv", index=False)
q2.style.format({"p_value": "{:.1e}"})

,feature,H,eta_sq_H,effect,p_value,median_Gram+ only,median_Gram- only,median_Gram+ & Gram-,median_Fungus only,median_Virus only,median_Cancer only
0,length,152.400000,0.047000,small,4.2e-31,13.000000,16.000000,18.000000,12.000000,18.000000,8.000000
1,net_charge,529.100000,0.169000,large,4.2e-112,2.760000,3.910000,3.010000,2.850000,-0.210000,-0.200000
2,charge_per_residue,526.400000,0.168000,large,1.6e-111,0.190000,0.270000,0.180000,0.170000,-0.010000,-0.020000
3,gravy,199.800000,0.063000,medium,3.1e-41,-0.270000,-0.930000,-0.180000,-0.680000,-0.130000,0.870000


In [11]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, f in zip(axes, FEATURES):
    data = [t.loc[t.target_group == g, f] for g in GROUPS]
    bp = ax.boxplot(data, tick_labels=GROUPS, showfliers=False, patch_artist=True, medianprops={"color": "black"})
    for patch in bp["boxes"]:
        patch.set_facecolor("#8ecae6")
    ax.set_title(TITLES[f], fontsize=10)
    ax.tick_params(axis="x", rotation=40)
    for lab in ax.get_xticklabels():
        lab.set_ha("right")
plt.tight_layout()
plt.savefig(f"{FIG}/q2_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 2b. Which pairs of groups differ?
Pairwise Cliff's delta (row group vs column group; positive = the row group has higher values), the pairwise effect sizes recommended after a Kruskal-Wallis test (Fiel Peres 2025).

In [12]:
pair_feats = ["length", "charge_per_residue", "gravy"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw={"wspace": 0.08})
pair_rows = []
for ax, f in zip(axes, pair_feats):
    M = np.zeros((len(GROUPS), len(GROUPS)))
    for i, j in combinations(range(len(GROUPS)), 2):
        d, p = cliffs_delta(t.loc[t.target_group == GROUPS[i], f], t.loc[t.target_group == GROUPS[j], f])
        M[i, j], M[j, i] = d, -d
        pair_rows.append({"feature": f, "group_1": GROUPS[i], "group_2": GROUPS[j],
                          "cliffs_delta": round(d, 3), "effect": size_label(d)})
    im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(GROUPS))); ax.set_xticklabels(GROUPS, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(GROUPS)))
    ax.set_yticklabels(GROUPS if ax is axes[0] else [], fontsize=8)   # labels only on the first panel
    for i in range(len(GROUPS)):
        for j in range(len(GROUPS)):
            if i != j:
                ax.text(j, i, f"{M[i, j]:+.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(M[i, j]) > 0.6 else "black")
    ax.set_title(TITLES[f], fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.8, label="Cliff's delta (row vs column)")
plt.savefig(f"{FIG}/q2_pairwise_delta.png", dpi=150, bbox_inches="tight")
plt.show()
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(f"{RES}/comparison_q2_pairwise.csv", index=False)
print("pairs with at least a medium effect (|delta| >= 0.28):")
pairs[pairs.cliffs_delta.abs() >= 0.28].sort_values("cliffs_delta", key=abs, ascending=False).head(12)

pairs with at least a medium effect (|delta| >= 0.28):


,feature,group_1,group_2,cliffs_delta,effect
23,charge_per_residue,Gram- only,Cancer only,0.787,large
22,charge_per_residue,Gram- only,Virus only,0.767,large
38,gravy,Gram- only,Cancer only,-0.668,large
43,gravy,Fungus only,Cancer only,-0.657,large
26,charge_per_residue,Gram+ & Gram-,Cancer only,0.606,large
25,charge_per_residue,Gram+ & Gram-,Virus only,0.579,large
19,charge_per_residue,Gram+ only,Cancer only,0.573,large
18,charge_per_residue,Gram+ only,Virus only,0.563,large
44,gravy,Virus only,Cancer only,-0.530,large
28,charge_per_residue,Fungus only,Cancer only,0.528,large


## 2c. Check: is this just Natural vs Synthetic again?
The origin mix differs a lot between the groups (see above). Since origin has a large effect on charge per residue (Q1), the Q2 differences could partly be an origin effect. As a sensitivity check we repeat the medians using **Synthetic peptides only**.

In [13]:
syn = t[t.origin == "Synthetic"]
chk = syn.groupby("target_group")[["length", "charge_per_residue", "gravy"]].median().reindex(GROUPS)
chk.insert(0, "n", syn.groupby("target_group").size().reindex(GROUPS))
chk.round(2).to_csv(f"{RES}/comparison_q2_synthetic_only.csv")
chk.round(2)

,n,length,charge_per_residue,gravy
target_group,,,,
Gram+ only,363,11.0,0.25,-0.40
Gram- only,268,14.0,0.31,-0.97
Gram+ & Gram-,1318,16.0,0.22,-0.21
Fungus only,132,9.0,0.32,-0.85
Virus only,380,18.0,-0.02,-0.13
Cancer only,58,12.0,0.06,0.94


# Q3 – Peptides listed with mammalian cells as a target
`target_mammalian_cell = 1` means DBAASP lists mammalian cells among the peptide's target groups. DBAASP collects hemolytic and cytotoxicity data (Pirtskhalava et al. 2016), but it also includes tested peptides regardless of outcome (Pirtskhalava et al. 2021). So this column tells us that a peptide was **listed with mammalian cells**, not that it is toxic. Only peptides with target information are used.

In [14]:
m = df[df.has_target_info].copy()
m["mammalian"] = np.where(m.target_mammalian_cell == 1, "Listed", "Not listed")
print(m["mammalian"].value_counts(), "\n")

def q3_table(data, label):
    r = compare_two(data, "mammalian", "Listed", "Not listed", FEATURES)
    r.insert(0, "subset", label)
    return r

q3 = pd.concat([q3_table(m, "all"),
                q3_table(m[m.origin == "Synthetic"], "Synthetic only"),
                q3_table(m[m.origin == "Natural"], "Natural only")], ignore_index=True)
q3.to_csv(f"{RES}/comparison_q3_mammalian.csv", index=False)
q3.round(2).style.format({"p_value": "{:.1e}"})

mammalian
Listed        5469
Not listed    4355
Name: count, dtype: int64 



,subset,feature,median_Listed,median_Not listed,cliffs_delta,effect,p_value
0,all,length,16.000000,17.000000,-0.010000,negligible,2.3e-01
1,all,net_charge,3.760000,2.760000,0.190000,small,0.0e+00
2,all,charge_per_residue,0.250000,0.150000,0.230000,small,0.0e+00
3,all,gravy,-0.200000,-0.240000,0.010000,negligible,5.5e-01
4,Synthetic only,length,14.000000,14.000000,0.030000,negligible,1.0e-02
5,Synthetic only,net_charge,3.760000,2.830000,0.230000,small,0.0e+00
6,Synthetic only,charge_per_residue,0.300000,0.220000,0.230000,small,0.0e+00
7,Synthetic only,gravy,-0.330000,-0.360000,-0.010000,negligible,5.0e-01
8,Natural only,length,24.000000,30.000000,-0.190000,small,0.0e+00
9,Natural only,net_charge,2.740000,2.480000,0.040000,negligible,7.0e-02


In [15]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, f in zip(axes, FEATURES):
    data = [m.loc[m.mammalian == g, f] for g in ["Listed", "Not listed"]]
    bp = ax.boxplot(data, tick_labels=["Listed with\nmammalian cells", "Not\nlisted"], showfliers=False,
                    patch_artist=True, medianprops={"color": "black"})
    for patch, c in zip(bp["boxes"], ["#c1121f", "#adb5bd"]):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    d = q3[(q3.subset == "all") & (q3.feature == f)].iloc[0]
    ax.set_title(f"{TITLES[f]}\nδ = {d.cliffs_delta:+.2f} ({d.effect})", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG}/q3_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## One-page summary table

In [16]:
summary = pd.DataFrame([
    {"question": "Q1 Natural vs Synthetic", "feature": r.feature, "effect_size_delta": r.cliffs_delta, "effect": r.effect}
    for r in q1.itertuples()
] + [
    {"question": "Q3 listed with mammalian cells vs not", "feature": r.feature, "effect_size_delta": r.cliffs_delta, "effect": r.effect}
    for r in q3[q3.subset == "all"].itertuples()
])
summary.to_csv(f"{RES}/comparison_summary.csv", index=False)
summary

,question,feature,effect_size_delta,effect
0,Q1 Natural vs Synthetic,length,0.487,large
1,Q1 Natural vs Synthetic,net_charge,-0.257,small
2,Q1 Natural vs Synthetic,charge_per_residue,-0.588,large
3,Q1 Natural vs Synthetic,gravy,0.318,medium
4,Q3 listed with mammalian cells vs not,length,-0.014,negligible
5,Q3 listed with mammalian cells vs not,net_charge,0.186,small
6,Q3 listed with mammalian cells vs not,charge_per_residue,0.229,small
7,Q3 listed with mammalian cells vs not,gravy,0.007,negligible


# Findings and limitations

All results use the **non-redundant set** (CD-HIT, 90% identity): 2,301 natural and 7,650 synthetic peptides. Numbers are medians unless stated; δ is Cliff's delta with the Vargha-Delaney labels reported by Fiel Peres (2025): |δ| < 0.11 negligible, ≥ 0.11 small, ≥ 0.28 medium, ≥ 0.43 large. Everything is "statistically significant" at this sample size, so the discussion relies on effect sizes.

## Q1 – Natural vs Synthetic
* **Charge per residue is the clearest difference:** 0.09 (natural) vs 0.28 (synthetic), δ = −0.59 (large). It holds when only short (10–24 aa) peptides are compared (δ = −0.66) and when peptides with D-amino acids are removed (δ = −0.58), and it is the same before and after redundancy removal (δ = −0.59 both times).
* **Length:** natural peptides are longer, 25 vs 14 residues (δ = +0.49, large). Net charge at pH 7 differs less (2.7 vs 3.8, δ = −0.26, small) because natural peptides are longer.
* **Mean hydropathy (GRAVY):** natural peptides are less hydrophilic, 0.17 vs −0.35 (δ = +0.32, medium; +0.51 among short peptides only, +0.34 before redundancy removal). Note that in the Kyte-Doolittle scale lysine and arginine are among the most hydrophilic residues (−3.9 and −4.5), so K/R-rich synthetic peptides get a low GRAVY; charge and GRAVY are therefore not independent measures.
* **Effect of CD-HIT:** it kept 71% of natural but only 51% of synthetic peptides (synthetic entries are more redundant, in line with DBAASP's note that many entries are derivatives). Charge per residue is unchanged; the other effect sizes moved by 0.02–0.04 (net charge went from −0.28 to −0.26, crossing the small/medium border).
* **Listed target groups:** natural peptides are listed with fungi much more often (54% vs 22%) and with parasites (5.0% vs 0.9%); Gram+ 87% vs 78%. Viruses are listed more often for synthetic peptides (7.7% vs 4.4%).
* Amino-acid composition is analysed against background populations in notebook 05.

## Q2 – Different listed target groups (exclusive groups only)
Group sizes: Gram+ only 435, Gram- only 308, Gram+ & Gram- 1,673, Fungus only 192, Virus only 385, Cancer only 122.

* Groups differ most in **charge**: ordinal eta-squared η²ₕ = 0.17 (large) for net charge and for charge per residue, 0.06 (medium) for GRAVY and 0.05 (small) for length.
* **Virus-only** and **Cancer-only** peptides have a median net charge of about 0 (−0.2), compared with 2.8–3.9 for the antibacterial and antifungal groups. Pairwise charge-per-residue δ between those groups and Virus-only or Cancer-only is +0.52 to +0.79 (large).
* **Gram- only** peptides have the highest charge per residue (0.27) and a low GRAVY (−0.93; Fungus-only −0.68). **Cancer-only** peptides are the shortest (median 8) and the only group with a positive GRAVY (+0.87; GRAVY δ of Gram- only vs Cancer only = −0.67), but that group is small.
* **Check:** the origin mix differs by group (Virus-only is 1% natural, Cancer-only 53%). Repeating the medians for synthetic peptides only gives the same picture (charge per residue: Virus-only −0.02, Cancer-only 0.06, Gram- only 0.31; GRAVY: Cancer-only +0.94, Gram- only −0.97), so this is not only an origin effect. The synthetic Cancer-only group is small (n = 58).

## Q3 – Peptides listed with mammalian cells (5,469 listed vs 4,355 not listed)
* Peptides listed with mammalian cells have somewhat higher charge per residue (0.25 vs 0.15, δ = +0.23) and net charge (3.8 vs 2.8, δ = +0.19): small effects. Length (δ = −0.01) and GRAVY (δ = +0.01) are negligible.
* The charge effects appear within synthetic peptides (δ = +0.23 for both) and within natural peptides (charge per residue δ = +0.19) separately, so they are not caused by the origin mix.
* DBAASP includes tested peptides whether or not they were active (Pirtskhalava et al. 2021), so "listed with mammalian cells" does not mean "toxic". More cationic peptides are somewhat more likely to be listed, but the overlap is large.

## Limitations
* The peptides are not a random sample of all peptides; they are those researchers chose to test and report in DBAASP. Synthetic peptides also outnumber natural ones about 3 to 1 even after redundancy removal.
* CD-HIT keeps the longest sequence of each cluster and merges shorter peptides that match part of a longer one, so the non-redundant set is biased towards longer peptides. Sequences with unusual or modified residues (`X`) were removed earlier (notebook 01).
* Target groups overlap, so Q2 uses only peptides with exactly one listed target set (or the Gram+ & Gram- pair). This leaves a fraction of the data and small groups for some targets.
* Target groups mean "listed with", not necessarily "active against" (see notebook 01).
* Net charge and GRAVY are calculated from the sequence alone: terminal modifications and D-amino-acid stereochemistry are ignored, and the values depend on the pH (7.0) and the scales used by Biopython.
* The effect-size labels are heuristics (Fiel Peres 2025), not sharp boundaries.
* These are associations in a curated database, not evidence that a feature *causes* a difference in activity.

## References
* Fiel Peres F (2025) Effect sizes for nonparametric tests. *Biochem Med (Zagreb)* 36(1):010101. https://doi.org/10.11613/BM.2026.010101 (PMID 41399660). Read in full; source of the Cliff's delta / Vargha-Delaney definitions and thresholds, the Kruskal-Wallis ordinal eta-squared formula and thresholds, and the pairwise-effect-size recommendation.
* Mann HB, Whitney DR (1947) On a test of whether one of two random variables is stochastically larger than the other. *Ann Math Stat* 18(1):50-60 (as cited in the SciPy documentation of `scipy.stats.mannwhitneyu`).
* Kruskal WH, Wallis WA (1952) Use of ranks in one-criterion variance analysis. *J Am Stat Assoc* 47(260):583-621 (as cited in the SciPy documentation of `scipy.stats.kruskal`).
* Pirtskhalava M, et al. (2016) DBAASP v.2. *Nucleic Acids Res* 44(D1):D1104-D1112. https://doi.org/10.1093/nar/gkv1174 (PMID 26578581)
* Pirtskhalava M, et al. (2021) DBAASP v3. *Nucleic Acids Res* 49(D1):D288-D297. https://doi.org/10.1093/nar/gkaa991 (PMID 33151284)